**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Optimization

Every 'fit', 'train', and 'design' in this curriculum is secretly `argmin`. Four sessions on why gradient methods work, when they're guaranteed to, what constraints do, and why stochastic noise is a feature — the theory under [ANN's](../../Intro_Mach_Learn/Intro_ANN/Intro_ANN.ipynb) backprop loop and [Adaptive Filtering's](../../Intro_Time_Series/Intro_AdFilt_APA.ipynb) LMS.

## 0. Introduction

The object of study: $\min_x f(x)$. Three questions organize the course — *when is the minimum unique?* (convexity), *how fast do we get there?* (rates), *what if $x$ is constrained?* (Lagrange).

## 1. Pre-requisites

[Linear Algebra](../Linear_Algebra/Linear_Algebra.ipynb) — especially S3 (eigenvalues) and S5 (matrix calculus). Multivariable calculus at the chain-rule level.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)

---
### 🕐 Session 1 of 4 — *Convexity* (~35 min)
**Goal:** recognize the functions where local = global; test convexity via the Hessian.
**Builds on:** [Linear Algebra](../Linear_Algebra/Linear_Algebra.ipynb). &nbsp; **Feeds into:** Session 2 (gradient descent).

---

## 2. Convex Functions

💡 **Intuition.** A convex function is a **bowl**: the chord between any two points sits above the graph. The consequence worth the whole session: *any local minimum is the global minimum* — a downhill walker cannot get trapped. Convex problems are the ones optimization can truly promise to solve; everything else (deep learning included) lives on borrowed intuition from this case.

**Definition.** $f$ is convex if for all $x, y$ and $\theta \in [0,1]$:
$$f(\theta x + (1-\theta) y) \le \theta f(x) + (1-\theta) f(y)$$

**Second-order test.** Twice-differentiable $f$ is convex iff its Hessian $\nabla^2 f \succeq 0$ (all eigenvalues $\ge 0$) everywhere — the bowl curves up along every axis. For a quadratic $f = \tfrac12 x^T S x - b^T x$, the Hessian *is* $S$: convex iff $S \succeq 0$, strictly (unique minimum) iff $S \succ 0$.

**Proof that local ⇒ global (convex case).** Let $x^\*$ be a local min and suppose $f(y) < f(x^\*)$ for some $y$. Points $\theta y + (1-\theta)x^\*$ approach $x^\*$ as $\theta \to 0$, and convexity gives $f(\theta y + (1-\theta)x^\*) \le \theta f(y) + (1-\theta) f(x^\*) < f(x^\*)$ — arbitrarily close points with lower value, contradicting local minimality. $\blacksquare$

In [2]:
# Convex vs nonconvex, and what a walker experiences
x = np.linspace(-3, 3, 400)
fig, axes = plt.subplots(1, 2, figsize=(9, 2.6))
axes[0].plot(x, x**2 + 0.5 * x); axes[0].set_title("convex: one basin, no traps")
axes[1].plot(x, x**4 - 3 * x**2 + x); axes[1].set_title("nonconvex: local minima exist")
for ax in axes: ax.grid(True)
plt.tight_layout(); plt.show()

/tmp/ipykernel_2012565/3997848261.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**Convexity zoo you already use:** least squares ($A^TA \succeq 0$), cross-entropy of a linear model, every norm ($\|\cdot\|_1, \|\cdot\|_2$), max of convex functions. **Not convex:** neural network losses — yet Session 4 explains why training works anyway.

---
### 🕐 Session 2 of 4 — *Gradient Descent & Rates* (~40 min)
**Goal:** prove how fast GD converges on smooth/strongly-convex problems; see conditioning bite.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (constraints).

---

## 3. Gradient Descent

💡 **Intuition.** GD's guarantee needs two numbers: $L$ (curvature never exceeds $L$ — you can trust the slope for a step of about $1/L$) and $\mu$ (curvature never below $\mu$ — the bowl never flattens out). Their ratio $\kappa = L/\mu$, the **condition number**, is the villain of the course: it is the elongation of the bowl, and error shrinks by a factor $\approx (1 - 1/\kappa)$ per step. Round bowl ⇒ sprint; canyon ⇒ zigzag crawl. For quadratics, $L$ and $\mu$ are just the extreme eigenvalues from [Linear Algebra S3](../Linear_Algebra/Linear_Algebra.ipynb).

**Theorem (rate, stated).** For $L$-smooth, $\mu$-strongly-convex $f$, GD with step $\eta = 1/L$ satisfies
$$\|x_k - x^*\|^2 \le \big(1 - \tfrac{\mu}{L}\big)^k \, \|x_0 - x^*\|^2$$
— *linear convergence*: a fixed fraction of the remaining error removed per step. (Proof for quadratics is a two-line eigen-argument: the error multiplies by $I - \eta S$, whose eigenvalues are $1 - \eta \lambda_i$.)

In [3]:
# Watch κ control the speed, exactly as the theorem says
def gd_path(S, x0, eta, steps=60):
    xs = [np.array(x0, float)]
    for _ in range(steps):
        xs.append(xs[-1] - eta * S @ xs[-1])
    return np.array(xs)

fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.4))
for ax, (l1, l2) in zip(axes, [(1.0, 2.0), (1.0, 25.0)]):
    S = np.diag([l1, l2])
    path = gd_path(S, [2.6, 1.8], eta=1/l2)
    g = np.linspace(-3, 3, 100)
    GX, GY = np.meshgrid(g, g)
    ax.contour(GX, GY, l1*GX**2/2 + l2*GY**2/2, levels=12, alpha=0.5)
    ax.plot(*path.T, "o-", markersize=2.5, color="crimson")
    kappa = l2 / l1
    ax.set_title(f"κ = {kappa:.0f}: {'sprints' if kappa < 5 else 'zigzags'}")
plt.tight_layout(); plt.show()

/tmp/ipykernel_2012565/913284460.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [4]:
# Measured rate vs predicted (1 − μ/L) per step
l1, l2 = 1.0, 25.0
path = gd_path(np.diag([l1, l2]), [2.6, 1.8], eta=1/l2, steps=200)
err = np.linalg.norm(path, axis=1)
measured = (err[-1] / err[100]) ** (1 / 100)
print(f"measured per-step factor {measured:.4f}   theory 1 − μ/L = {1 - l1/l2:.4f}")

measured per-step factor 0.9600   theory 1 − μ/L = 0.9600


This is *why* preconditioning, momentum, and Adam exist: they all attack $\kappa$. And it's the same $\lambda_{max}$ speed limit you met as $\mu < 2/\lambda_{max}$ in [LMS](../../Intro_Time_Series/Intro_AdFilt_APA.ipynb).

---
### 🕐 Session 3 of 4 — *Constraints: Lagrange & KKT* (~35 min)
**Goal:** optimize with equality and inequality constraints; read a Lagrangian like a force balance.
**Builds on:** Session 2. &nbsp; **Feeds into:** Session 4 (SGD).

---

## 4. Constrained Optimization

💡 **Intuition.** At a constrained optimum you cannot improve *without leaving the feasible set* — so the objective's downhill direction must be exactly opposed by the constraint's 'wall'. Algebraically: $\nabla f$ is a combination of constraint normals. The multipliers $\lambda$ are the strengths of the wall forces — and economically, the *price* of tightening each constraint.

**Lagrange (equality).** For $\min f$ s.t. $g(x) = 0$: at a (regular) optimum $\exists \lambda$ with $\nabla f = \lambda \nabla g$. Solve by stationarity of $\mathcal{L}(x, \lambda) = f(x) - \lambda g(x)$.

**KKT (inequality, $h(x) \le 0$)** adds two conditions: $\lambda \ge 0$ (walls only push, never pull) and *complementary slackness* $\lambda h(x) = 0$ (an inactive constraint exerts no force).

**Worked example.** $\min\; x^T S x$ s.t. $\|x\| = 1$: stationarity gives $S x = \lambda x$ — the constrained minimizer is the **smallest eigenvector**. Eigenproblems *are* constrained optimization; this is how PCA, MVDR beamforming ([Array Processing](../../Intro_DSP/Array_Processing.ipynb)), and Rayleigh quotients arise.

In [5]:
# Verify: minimize xᵀSx on the unit circle — numerically vs the eigen-answer
M = rng.standard_normal((2, 2)); S = M @ M.T + 0.1 * np.eye(2)
th = np.linspace(0, 2 * np.pi, 2000)
circle = np.stack([np.cos(th), np.sin(th)])
vals = np.einsum("ij,ji->i", circle.T @ S, circle)
x_num = circle[:, vals.argmin()]

w, V = np.linalg.eigh(S)
print("numerical minimizer:", np.round(x_num, 4), " value", vals.min().round(4))
print("smallest eigenvector:", np.round(V[:, 0], 4), " eigenvalue", w[0].round(4))

numerical minimizer: [-0.9864  0.1643]  value 0.1221
smallest eigenvector: [-0.9863  0.1648]  eigenvalue 0.1221


---
### 🕐 Session 4 of 4 — *Stochastic Gradient Descent* (~40 min)
**Goal:** understand SGD's noise: why it's cheap, why it still converges, and why it can even help.
**Builds on:** Sessions 2–3.

---

## 5. SGD

💡 **Intuition.** Full gradients cost a pass over ALL data; SGD gambles on a mini-batch's estimate. The estimate is **unbiased** — right on average — so each step is downhill *in expectation*, and averaging over steps mimics averaging over data (the LLN from [Independence](../Analysis/Independence.ipynb)). The price: a noise floor set by step size × gradient variance. The classic cure: **decay the step size** — big steps to travel, small steps to settle. And in nonconvex landscapes the noise moonlights as an explorer, rattling the iterate out of narrow bad minima.

**The trade in one equation** (strongly convex case, stated): with constant step $\eta$,
$$E\|x_k - x^*\|^2 \lesssim \underbrace{(1 - \eta\mu)^k \|x_0 - x^*\|^2}_{\text{bias: shrinks}} + \underbrace{\frac{\eta \, \sigma^2}{\mu}}_{\text{noise floor: doesn't}}$$
Decaying $\eta_k \propto 1/k$ drives both terms to zero (at the slower $O(1/k)$ rate).

In [6]:
# See the noise floor and the decay cure, on least squares
A = rng.standard_normal((2000, 20)); x_true = rng.standard_normal(20)
b = A @ x_true + 0.5 * rng.standard_normal(2000)
x_star, *_ = np.linalg.lstsq(A, b, rcond=None)

def sgd(eta_fn, steps=8000, batch=8):
    x = np.zeros(20); errs = []
    for k in range(steps):
        i = rng.integers(0, len(A), batch)
        g = 2 * A[i].T @ (A[i] @ x - b[i]) / batch
        x -= eta_fn(k) * g
        if k % 20 == 0: errs.append(np.linalg.norm(x - x_star))
    return np.array(errs)

plt.figure(figsize=(8, 3))
for label, fn in [("η = 0.01 (floor!)", lambda k: 0.01),
                  ("η = 0.001 (lower floor, slower)", lambda k: 0.001),
                  ("η = 0.01/(1+k/1000) (decay: best of both)", lambda k: 0.01 / (1 + k / 1000))]:
    plt.semilogy(np.arange(0, 8000, 20), sgd(fn), label=label, alpha=0.8)
plt.legend(); plt.grid(True, alpha=0.3)
plt.xlabel("iteration"); plt.ylabel("‖x − x*‖")
plt.title("SGD: converge fast OR settle low — decay schedules buy both")
plt.tight_layout(); plt.show()

/tmp/ipykernel_2012565/4068096982.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 6. Conclusion

Convexity is the promise, $\kappa$ the speed limit, Lagrange the wall forces, and SGD the affordable gamble with a noise floor you now know how to lower. Deep-learning practice ([Training Dynamics](../../Intro_Mach_Learn/Training_Dynamics.ipynb)) is engineering against exactly these quantities.

---
## Where next

- [Training Dynamics](../../Intro_Mach_Learn/Training_Dynamics.ipynb) — Adam, schedules, and regularization as applied versions of these ideas.
- [Adaptive Filtering](../../Intro_Time_Series/Intro_AdFilt_APA.ipynb) — SGD in real time, under the name LMS.
- [Estimation Theory](../Estimation_Theory/Estimation_Theory.ipynb) — what the minimum *means* statistically.